# Assignment 4 - Document Similarity & Topic Modelling

## Part 1 - Document Similarity

For the first part of this assignment, you will complete the functions `doc_to_synsets` and `similarity_score` which will be used by `document_path_similarity` to find the path similarity between two documents.

The following functions are provided:
* **`convert_tag:`** converts the tag given by `nltk.pos_tag` to a tag used by `wordnet.synsets`. You will need to use this function in `doc_to_synsets`.
* **`document_path_similarity:`** computes the symmetrical path similarity between two documents by finding the synsets in each document using `doc_to_synsets`, then computing similarities using `similarity_score`.

You will need to finish writing the following functions:
* **`doc_to_synsets:`** returns a list of synsets in document. This function should first tokenize and part of speech tag the document using `nltk.word_tokenize` and `nltk.pos_tag`. Then it should find each tokens corresponding synset using `wn.synsets(token, wordnet_tag)`. The first synset match should be used. If there is no match, that token is skipped.
* **`similarity_score:`** returns the normalized similarity score of a list of synsets (s1) onto a second list of synsets (s2). For each synset in s1, find the synset in s2 with the largest similarity value. Sum all of the largest similarity values together and normalize this value by dividing it by the number of largest similarity values found. Be careful with data types, which should be floats. Missing values should be ignored.

Once doc_to_synsets and similarity_score have been completed, submit to the autograder which will run a test to check that these functions are running correctly.

*Do not modify the functions `convert_tag` and `document_path_similarity`.*

### แนวคิด: Semantic Similarity

WordNet จัดคำที่มีความหมายเดียวกันเป็น **synset** และเชื่อม synset ด้วยความสัมพันธ์ เช่น hypernym และ hyponym งานนี้ใช้ path similarity ซึ่งให้คะแนนสูงเมื่อเส้นทางระหว่างสอง synset สั้น

ขั้นตอนคือ tokenize ข้อความ ทำ POS tagging เลือก synset แรกของแต่ละคำ แล้วหาคะแนนที่ดีที่สุดของทุก synset แบบสองทิศทาง การเฉลี่ยสองทิศทางทำให้ผลเป็นแบบสมมาตรแม้เอกสารมีจำนวนคำต่างกัน

In [1]:
%%capture
import importlib.util
import subprocess
import sys

# ติดตั้งเฉพาะไลบรารีที่ Colab runtime ยังไม่มี
required_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "nltk": "nltk",
    "gensim": "gensim",
}
missing_packages = [
    pip_name
    for module_name, pip_name in required_packages.items()
    if importlib.util.find_spec(module_name) is None
]
if missing_packages:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *missing_packages]
    )

import numpy as np
import nltk
import pandas as pd
from nltk.corpus import wordnet as wn

for resource in [
    "punkt",
    "punkt_tab",
    "averaged_perceptron_tagger",
    "averaged_perceptron_tagger_eng",
    "wordnet",
    "omw-1.4",
]:
    nltk.download(resource, quiet=True)


def convert_tag(tag):
    """Convert the tag given by nltk.pos_tag to the tag used by wordnet.synsets"""

    tag_dict = {'N': 'n', 'J': 'a', 'R': 'r', 'V': 'v'}
    try:
        return tag_dict[tag[0]]
    except KeyError:
        return None

In [2]:
def doc_to_synsets(doc):
    """
    Returns a list of synsets in document.

    Tokenizes and tags the words in the document doc.
    Then finds the first synset for each word/tag combination.
    If a synset is not found for that combination it is skipped.

    Args:
        doc: string to be converted

    Returns:
        list of synsets

    Example:
        doc_to_synsets('Fish are friends.')
        Out: [Synset('fish.n.01'), Synset('be.v.01'), Synset('friend.n.01')]
    """

    # YOUR CODE HERE
    tokens = nltk.word_tokenize(doc)
    tagged_tokens = nltk.pos_tag(tokens)
    synsets = []

    for token, tag in tagged_tokens:
        wordnet_tag = convert_tag(tag)
        if wordnet_tag is None:
            continue

        # ส่วนสำคัญ: จำกัด POS ก่อนเลือก synset แรกตามเงื่อนไขของโจทย์
        matches = wn.synsets(token, pos=wordnet_tag)
        if matches:
            synsets.append(matches[0])

    return synsets  # Your Answer Here


def similarity_score(s1, s2):
    """
    Calculate the normalized similarity score of s1 onto s2

    For each synset in s1, finds the synset in s2 with the largest similarity value.
    Sum of all of the largest similarity values and normalize this value by dividing it by the
    number of largest similarity values found.

    Args:
        s1, s2: list of synsets from doc_to_synsets

    Returns:
        normalized similarity score of s1 onto s2

    Example:
        synsets1 = doc_to_synsets('I like cats')
        synsets2 = doc_to_synsets('I like dogs')
        similarity_score(synsets1, synsets2)
        Out: 0.7333333333333333
    """

    # YOUR CODE HERE
    best_scores = []

    for synset1 in s1:
        similarities = [
            synset1.path_similarity(synset2)
            for synset2 in s2
        ]
        valid_scores = [
            score for score in similarities
            if score is not None
        ]
        if valid_scores:
            best_scores.append(max(valid_scores))

    # คืน 0.0 เมื่อไม่มี synset ที่เปรียบเทียบได้ เพื่อป้องกันการหารด้วยศูนย์
    return float(np.mean(best_scores)) if best_scores else 0.0  # Your Answer Here

In [3]:
def document_path_similarity(doc1, doc2):
    """Finds the symmetrical similarity between doc1 and doc2"""

    synsets1 = doc_to_synsets(doc1)
    synsets2 = doc_to_synsets(doc2)

    return (similarity_score(synsets1, synsets2) + similarity_score(synsets2, synsets1)) / 2

In [4]:
from pathlib import Path

required_files = {"paraphrases.csv", "newsgroups"}


def find_data_dir():
    candidates = [Path.cwd(), Path.cwd() / "Assignment 4", Path("/content")]
    for candidate in candidates:
        if all((candidate / filename).exists() for filename in required_files):
            return candidate
    return None


DATA_DIR = find_data_dir()

# บน Colab จะเปิดหน้าต่างอัปโหลดเฉพาะเมื่อยังไม่พบไฟล์ข้อมูล
if DATA_DIR is None:
    try:
        from google.colab import files

        print("กรุณาอัปโหลด paraphrases.csv และ newsgroups พร้อมกัน")
        files.upload()
        DATA_DIR = find_data_dir()
    except ImportError:
        pass

if DATA_DIR is None:
    raise FileNotFoundError("ไม่พบ paraphrases.csv และ newsgroups")

print(f"Data directory: {DATA_DIR.resolve()}")

Data directory: E:\AIforSocial\Ajpich\Assignment 4


`paraphrases` is a DataFrame which contains the following columns: `Quality`, `D1`, and `D2`.

`Quality` is an indicator variable which indicates if the two documents `D1` and `D2` are paraphrases of one another (1 for paraphrase, 0 for not paraphrase).

In [5]:
# Use this dataframe for questions most_similar_docs and label_accuracy
paraphrases = pd.read_csv(DATA_DIR / 'paraphrases.csv')
paraphrases.head()

,Quality,D1,D2
0,1,"Ms Stewart, the chief executive, was not expec...","Ms Stewart, 61, its chief executive officer an..."
1,1,After more than two years' detention under the...,After more than two years in detention by the ...
2,1,"""It still remains to be seen whether the reven...","""It remains to be seen whether the revenue rec..."
3,0,"And it's going to be a wild ride,"" said Allan ...","Now the rest is just mechanical,"" said Allan H..."
4,1,The cards are issued by Mexico's consulates to...,The card is issued by Mexico's consulates to i...


### most_similar_docs

Using `document_path_similarity`, find the pair of documents in paraphrases which has the maximum similarity score.

*This function should return a tuple `(D1, D2, similarity_score)`*

In [6]:
def most_similar_docs():

    # YOUR CODE HERE
    similarities = paraphrases.apply(
        lambda row: document_path_similarity(row['D1'], row['D2']),
        axis=1
    )
    max_index = similarities.idxmax()

    return (
        paraphrases.loc[max_index, 'D1'],
        paraphrases.loc[max_index, 'D2'],
        float(similarities.loc[max_index])
    )  # Your Answer Here


most_similar_docs()

('"Indeed, Iran should be put on notice that efforts to try to remake Iraq in their image will be aggressively put down," he said.',
 '"Iran should be on notice that attempts to remake Iraq in Iran\'s image will be aggressively put down," he said.\n',
 0.9481481481481482)

### คำอธิบายผล `most_similar_docs`

คู่เอกสารที่คล้ายที่สุดกล่าวถึงการเตือนอิหร่านไม่ให้พยายามเปลี่ยนอิรักตามแบบของอิหร่านเหมือนกันทั้งสองประโยค และได้ similarity ประมาณ **0.9481**

คะแนนสูงเพราะคำและแนวคิดหลัก เช่น `Iran`, `notice`, `remake`, `Iraq`, `image` และ `aggressively put down` ตรงกันเกือบทั้งหมด ความแตกต่างส่วนใหญ่เป็นเพียงการใช้ “efforts to try to” แทน “attempts to” จึงมีลักษณะเป็น paraphrase ชัดเจน

### label_accuracy

Provide labels for the twenty pairs of documents by computing the similarity for each pair using `document_path_similarity`. Let the classifier rule be that if the score is greater than 0.75, label is paraphrase (1), else label is paraphrase (0). Report accuracy of the classifier using scikit-learn's accuracy_score.

*This function should return a float.*

In [7]:
def label_accuracy():
    from sklearn.metrics import accuracy_score

    # YOUR CODE HERE
    similarities = paraphrases.apply(
        lambda row: document_path_similarity(row['D1'], row['D2']),
        axis=1
    )

    # ส่วนสำคัญ: โจทย์กำหนดให้คะแนนมากกว่า 0.75 เท่านั้นเป็น paraphrase
    predicted_labels = (similarities > 0.75).astype(int)

    return float(
        accuracy_score(paraphrases['Quality'], predicted_labels)
    )  # Your Answer Here


label_accuracy()

0.7

### คำอธิบายผล `label_accuracy`

Accuracy เท่ากับ **0.70** หรือจำแนกถูก 14 จาก 20 คู่ กฎ threshold 0.75 ตรวจคู่ที่ไม่ใช่ paraphrase ได้ค่อนข้างดี แต่พลาดคู่ที่เป็น paraphrase หลายคู่

สาเหตุคือ WordNet path similarity ใช้ synset แรกของคำ ไม่พิจารณาลำดับคำ บริบท การปฏิเสธ หรือความหมายรวมของประโยค และค่า 0.75 เป็น threshold ที่กำหนดไว้ ไม่ได้ปรับจาก validation set ผลนี้จึงเหมาะเป็น baseline ที่อธิบายได้ง่าย

## Part 2 - Topic Modelling

For the second part of this assignment, you will use Gensim's LDA (Latent Dirichlet Allocation) model to model topics in `newsgroup_data`. You will first need to finish the code in the cell below by using gensim.models.ldamodel.LdaModel constructor to estimate LDA model parameters on the corpus, and save to the variable `ldamodel`. Extract 10 topics using `corpus` and `id_map`, and with `passes=25` and `random_state=34`.

### แนวคิด: Topic Modeling, Generative Models และ LDA

Topic modeling เป็นการเรียนรู้แบบไม่มีผู้สอนเพื่อค้นหาหัวข้อที่ซ่อนในเอกสาร โดย **หัวข้อเป็นการแจกแจงความน่าจะเป็นเหนือคำ** และ **เอกสารหนึ่งฉบับเป็นส่วนผสมของหลายหัวข้อ**

LDA เป็น generative model ที่สมมติว่าเอกสารเกิดจากการเลือกสัดส่วนหัวข้อ แล้วเลือกหัวข้อและคำในแต่ละตำแหน่ง ตอนฝึกโมเดลจึงย้อนจากคำที่สังเกตพบเพื่อประมาณ word distribution ของแต่ละหัวข้อและ topic distribution ของแต่ละเอกสาร

In [8]:
import pickle
import gensim
from sklearn.feature_extraction.text import CountVectorizer

# Load the list of documents
with open(DATA_DIR / 'newsgroups', 'rb') as f:
    newsgroup_data = pickle.load(f)

# Use CountVectorizor to find three letter tokens, remove stop_words,
# remove tokens that don't appear in at least 20 documents,
# remove tokens that appear in more than 20% of the documents
vect = CountVectorizer(min_df=20, max_df=0.2, stop_words='english',
                       token_pattern='(?u)\\b\\w\\w\\w+\\b')
# Fit and transform
X = vect.fit_transform(newsgroup_data)

# Convert sparse matrix to gensim corpus.
corpus = gensim.matutils.Sparse2Corpus(X, documents_columns=False)

# Mapping from word IDs to words (To be used in LdaModel's id2word parameter)
id_map = dict((v, k) for k, v in vect.vocabulary_.items())

In [9]:
# Use the gensim.models.ldamodel.LdaModel constructor to estimate
# LDA model parameters on the corpus, and save to the variable `ldamodel`

ldamodel = None
# YOUR CODE HERE
# random_state ทำให้ผลจากการสุ่มเริ่มต้นของ LDA ทำซ้ำได้
ldamodel = gensim.models.ldamodel.LdaModel(
    corpus=corpus,
    num_topics=10,
    id2word=id_map,
    passes=25,
    random_state=34
)

### lda_topics

Using `ldamodel`, find a list of the 10 topics and the most significant 10 words in each topic. This should be structured as a list of 10 tuples where each tuple takes on the form:

`(9, '0.068*"space" + 0.036*"nasa" + 0.021*"science" + 0.020*"edu" + 0.019*"data" + 0.017*"shuttle" + 0.015*"launch" + 0.015*"available" + 0.014*"center" + 0.013*"information"')`

for example.

*This function should return a list of tuples.*

In [10]:
def lda_topics():

    # YOUR CODE HERE
    return ldamodel.print_topics(
        num_topics=10,
        num_words=10
    )  # Your Answer Here

In [11]:
lda_topics()

[(0,
  '0.056*"edu" + 0.043*"com" + 0.033*"thanks" + 0.022*"mail" + 0.021*"know" + 0.020*"does" + 0.014*"info" + 0.012*"monitor" + 0.010*"looking" + 0.010*"don"'),
 (1,
  '0.024*"ground" + 0.018*"current" + 0.018*"just" + 0.013*"want" + 0.013*"use" + 0.011*"using" + 0.011*"used" + 0.010*"power" + 0.010*"speed" + 0.010*"output"'),
 (2,
  '0.061*"drive" + 0.042*"disk" + 0.033*"scsi" + 0.030*"drives" + 0.028*"hard" + 0.028*"controller" + 0.027*"card" + 0.020*"rom" + 0.018*"floppy" + 0.017*"bus"'),
 (3,
  '0.023*"time" + 0.015*"atheism" + 0.014*"list" + 0.013*"left" + 0.012*"alt" + 0.012*"faq" + 0.012*"probably" + 0.011*"know" + 0.011*"send" + 0.010*"months"'),
 (4,
  '0.025*"car" + 0.016*"just" + 0.014*"don" + 0.014*"bike" + 0.012*"good" + 0.011*"new" + 0.011*"think" + 0.010*"year" + 0.010*"cars" + 0.010*"time"'),
 (5,
  '0.030*"game" + 0.027*"team" + 0.023*"year" + 0.017*"games" + 0.016*"play" + 0.012*"season" + 0.012*"players" + 0.012*"win" + 0.011*"hockey" + 0.011*"good"'),
 (6,
  '0.0

### คำอธิบายผล `lda_topics`

คำสำคัญทำให้ตีความหัวข้อเด่นได้ เช่น

- Topic 2: `drive`, `disk`, `scsi`, `controller`, `floppy` เป็น Computers & IT
- Topic 4: `car`, `bike`, `cars` เป็น Automobiles
- Topic 5: `game`, `team`, `season`, `players`, `hockey` เป็น Sports
- Topic 6: `medical`, `research`, `university` มีแนวโน้มเป็น Health
- Topic 9: `space`, `nasa`, `science`, `shuttle`, `launch` เป็น Science

ชื่อหัวข้อไม่ได้ถูกสร้างโดย LDA โดยตรง แต่เป็นการตีความ word distribution ของผู้วิเคราะห์ จึงอาจมีบางหัวข้อที่ใช้ชื่อซ้ำหรือมีเนื้อหาคาบเกี่ยวกัน

### topic_distribution

For the new document `new_doc`, find the topic distribution. Remember to use vect.transform on the the new doc, and Sparse2Corpus to convert the sparse matrix to gensim corpus.

*This function should return a list of tuples, where each tuple is `(#topic, probability)`*

In [12]:
new_doc = ["\n\nIt's my understanding that the freezing will start to occur because \
of the\ngrowing distance of Pluto and Charon from the Sun, due to it's\nelliptical orbit. \
It is not due to shadowing effects. \n\n\nPluto can shadow Charon, and vice-versa.\n\nGeorge \
Krumins\n-- "]

In [13]:
def topic_distribution():

    # YOUR CODE HERE
    new_doc_matrix = vect.transform(new_doc)
    new_doc_corpus = gensim.matutils.Sparse2Corpus(
        new_doc_matrix,
        documents_columns=False
    )

    # ส่วนสำคัญ: ใช้ vectorizer เดิมเพื่อให้ word id ตรงกับโมเดล
    document_bow = next(iter(new_doc_corpus))
    distribution = ldamodel.get_document_topics(
        document_bow,
        minimum_probability=0.0
    )

    return [
        (topic_id, float(probability))
        for topic_id, probability in distribution
    ]  # Your Answer Here

In [14]:
topic_distribution()

[(0, 0.02000311017036438),
 (1, 0.02000332623720169),
 (2, 0.020001282915472984),
 (3, 0.49674853682518005),
 (4, 0.020004039630293846),
 (5, 0.02000413089990616),
 (6, 0.020002974197268486),
 (7, 0.02000264637172222),
 (8, 0.02000313065946102),
 (9, 0.34322690963745117)]

### คำอธิบายผล `topic_distribution`

ผลจากข้อความ `new_doc` ในแม่แบบเดิมให้ **Topic 3** สูงสุดประมาณ **0.497** และ **Topic 9** รองลงมาประมาณ **0.343** ส่วนหัวข้ออื่นมีค่าประมาณ 0.020

Topic 9 มีคำสำคัญ `space`, `nasa`, `science`, `shuttle` และ `launch` จึงสัมพันธ์กับเนื้อหา Pluto, Charon, Sun และวงโคจรโดยตรง ส่วน Topic 3 มีคำทั่วไปและคำจากข้อความกลุ่มข่าวปะปนอยู่ จึงรับสัดส่วนสูงจากคำที่เหลือหลัง preprocessing

ผลนี้แสดงข้อจำกัดสำคัญของ LDA: คำเฉพาะอย่าง `Pluto` และ `Charon` อาจถูกตัดเพราะ `min_df=20` และโมเดลพิจารณา bag-of-words ไม่ได้เข้าใจความหมายทั้งประโยค จึงควรตีความ probability ร่วมกับ vocabulary และคำสำคัญของแต่ละหัวข้อ

### topic_names

From the list of the following given topics, assign topic names to the topics you found. If none of these names best matches the topics you found, create a new 1-3 word "title" for the topic.

Topics: Health, Science, Automobiles, Politics, Government, Travel, Computers & IT, Sports, Business, Society & Lifestyle, Religion, Education.

*This function should return a list of 10 strings.*

In [15]:
def topic_names():

    # YOUR CODE HERE
    return [
        'Computers & IT',
        'Computers & IT',
        'Computers & IT',
        'Religion',
        'Automobiles',
        'Sports',
        'Health',
        'Society & Lifestyle',
        'Computers & IT',
        'Science'
    ]  # Your Answer Here

In [16]:
topic_names()

['Computers & IT',
 'Computers & IT',
 'Computers & IT',
 'Religion',
 'Automobiles',
 'Sports',
 'Health',
 'Society & Lifestyle',
 'Computers & IT',
 'Science']

### ความเชื่อมโยงกับ Information Extraction

Information Extraction เปลี่ยนข้อความอิสระให้เป็นข้อมูลแบบมีโครงสร้าง เช่น PERSON, ORGANIZATION, LOCATION, วันที่ และความสัมพันธ์ระหว่าง entity โดย NER ต้องทั้งตรวจขอบเขตของ entity และจำแนกประเภท

Topic modeling ในงานนี้ช่วยบอกว่าเอกสารกล่าวถึงเรื่องใด ส่วน Information Extraction จะลงรายละเอียดต่อว่าในเอกสารมีใคร ที่ไหน เมื่อใด และมีความสัมพันธ์กันอย่างไร ฟิลด์ที่มีรูปแบบแน่นอนอาจใช้ regular expression ส่วน entity ที่ซับซ้อนมักใช้พจนานุกรมหรือ supervised machine learning